## 🎯 Learning Objectives
* Understand the fundamental concepts of multi-step planning and task decomposition in AI agents.
* Learn why multi-step planning is crucial for handling complex, real-world problems beyond single-shot interactions.
* Implement a basic agentic loop demonstrating task decomposition and sequential execution of sub-tasks.
* Analyze the trade-offs and practical considerations when designing and deploying multi-step planning agents.


## AG03-L05: Multi-step Planning and Task Decomposition

Welcome to Lesson 5 of AG-03! In the previous lessons, we explored the foundational components of AI agents, including basic ReAct loops and tool integration. While these are powerful, real-world problems are rarely solved in a single, atomic step. Imagine trying to build a house by just saying "build a house" to a construction crew. It's impossible without a detailed plan.

This is where **multi-step planning** and **task decomposition** come into play. These are critical capabilities that elevate an AI agent from a reactive tool-user to a proactive problem-solver, enabling it to tackle complex, ambiguous, and long-horizon tasks.

### The Analogy: A Project Manager

Think of a human project manager. When given a large, overarching goal (e.g., "Launch a new product line"), they don't immediately jump into action. Instead, they:

1.  **Decompose the Goal**: Break down the main objective into smaller, more manageable sub-tasks (e.g., market research, product design, manufacturing, marketing campaign, sales strategy).
2.  **Sequence and Prioritize**: Determine the order in which these sub-tasks need to be completed, identifying dependencies (e.g., market research must precede product design).
3.  **Assign Resources/Tools**: Allocate specific teams or tools to each sub-task (e.g., design team for product design, marketing agency for campaigns).
4.  **Execute Iteratively**: Oversee the execution of each sub-task, monitoring progress.
5.  **Monitor and Re-plan**: Continuously check if the plan is working, adapt to unforeseen challenges, and adjust the plan as needed.

AI agents, especially those powered by advanced Large Language Models (LLMs) in 2026, adopt a similar methodology. Instead of a human project manager, the LLM acts as the central planner, reasoning about the best way to achieve a goal.

### Why is this crucial for AI Agents?

*   **Handling Complexity**: Real-world problems are often too complex for a single LLM prompt or tool call. Decomposition breaks them into digestible pieces.
*   **Overcoming Context Window Limitations**: LLMs have finite context windows. Breaking a task down allows the agent to focus on relevant information for each sub-task, preventing context overflow.
*   **Improved Reliability and Accuracy**: Smaller tasks are easier to execute correctly. Errors in one sub-task can be identified and corrected before they cascade.
*   **Enabling Specialized Tools**: Different sub-tasks might require different specialized tools. Planning allows the agent to select and use the right tool at the right time.
*   **Robustness and Error Recovery**: If a sub-task fails, the agent can re-plan or retry that specific step without having to restart the entire process.

### The Core Process

1.  **Initial Goal**: The agent receives a high-level objective.
2.  **Planning/Decomposition Module**: An LLM (or a rule-based system for simpler cases) analyzes the goal and generates a sequence of sub-tasks. This often involves reasoning about dependencies and potential tool calls.
3.  **Execution Loop**: The agent iterates through the planned sub-tasks.
4.  **Tool Invocation**: For each sub-task, the agent selects and invokes the appropriate tool(s).
5.  **Observation and State Update**: The agent observes the outcome of the tool call and updates its internal state or understanding of the problem.
6.  **Re-planning (Optional but Recommended)**: Based on observations, the agent might decide to refine the current plan, add new steps, or even abandon a path if it proves unfruitful. This makes the agent adaptive.

In this lesson, we'll build a simplified agent that demonstrates this multi-step planning and decomposition process, simulating an LLM's role in generating a plan and executing it using mock tools.


In [ ]:
import time
from typing import List, Dict, Callable, Any

# --- 1. Simulate an LLM for Task Decomposition ---
# In a real 2026 scenario, this would be an actual LLM call (e.g., via Google AI Studio's Gemini API)
# that takes a prompt and returns a structured plan.

def simulate_llm_decompose_task(goal: str) -> List[Dict[str, str]]:
    """
    Simulates an LLM's ability to decompose a high-level goal into sub-tasks.
    For simplicity, this is hardcoded based on the goal.
    """
    print(f"\n[LLM Planner]: Decomposing goal: '{goal}'...")
    time.sleep(0.5) # Simulate LLM processing time

    if "plan a trip to Paris for a conference" in goal.lower():
        return [
            {"task": "Search for conference dates and venue in Paris", "tool": "search_web"},
            {"task": "Find suitable flights to Paris", "tool": "book_flight"},
            {"task": "Research and book accommodation near the conference venue", "tool": "find_hotel"},
            {"task": "Register for the conference", "tool": "register_conference"},
            {"task": "Plan local transportation and activities", "tool": "search_web"}
        ]
    elif "write a blog post about AI agents" in goal.lower():
        return [
            {"task": "Research key concepts of AI agents", "tool": "search_web"},
            {"task": "Outline the blog post structure", "tool": "generate_outline"},
            {"task": "Draft the introduction and main sections", "tool": "write_draft"},
            {"task": "Review and refine the content", "tool": "review_text"},
            {"task": "Generate a catchy title and meta description", "tool": "generate_title"}
        ]
    else:
        return [
            {"task": f"Perform general research on: {goal}", "tool": "search_web"},
            {"task": "Summarize findings", "tool": "summarize_text"}
        ]

# --- 2. Define Mock Tools ---
# These functions simulate external services or APIs that an agent can call.

def search_web(query: str) -> str:
    """
    Simulates searching the web for information.
    """
    print(f"  [Tool: search_web] Searching for: '{query}'...")
    time.sleep(0.3)
    return f"Found relevant information for '{query}'."

def book_flight(destination: str, date_range: str) -> str:
    """
    Simulates booking a flight.
    """
    print(f"  [Tool: book_flight] Booking flight to {destination} for {date_range}...")
    time.sleep(0.7)
    return f"Flight to {destination} on {date_range} successfully booked."

def find_hotel(location: str, criteria: str) -> str:
    """
    Simulates finding and booking a hotel.
    """
    print(f"  [Tool: find_hotel] Finding hotel in {location} with criteria '{criteria}'...")
    time.sleep(0.6)
    return f"Hotel in {location} matching '{criteria}' found and reserved."

def register_conference(conference_name: str) -> str:
    """
    Simulates registering for a conference.
    """
    print(f"  [Tool: register_conference] Registering for '{conference_name}'...")
    time.sleep(0.4)
    return f"Successfully registered for '{conference_name}'."

def generate_outline(topic: str) -> str:
    """
    Simulates generating an outline for a given topic.
    """
    print(f"  [Tool: generate_outline] Generating outline for '{topic}'...")
    time.sleep(0.3)
    return f"Outline for '{topic}' generated: Intro, Key Concepts, Applications, Conclusion."

def write_draft(section: str) -> str:
    """
    Simulates writing a draft for a specific section.
    """
    print(f"  [Tool: write_draft] Drafting section: '{section}'...")
    time.sleep(0.8)
    return f"Draft for '{section}' completed."

def review_text(text: str) -> str:
    """
    Simulates reviewing and refining text.
    """
    print(f"  [Tool: review_text] Reviewing text for quality and coherence...")
    time.sleep(0.5)
    return f"Text reviewed and refined."

def generate_title(content_summary: str) -> str:
    """
    Simulates generating a title and meta description.
    """
    print(f"  [Tool: generate_title] Generating title for content: '{content_summary}'...")
    time.sleep(0.4)
    return f"Title and meta description generated."

def summarize_text(text: str) -> str:
    """
    Simulates summarizing text.
    """
    print(f"  [Tool: summarize_text] Summarizing text...")
    time.sleep(0.3)
    return f"Text summarized."

# Map tool names to their functions
TOOLS: Dict[str, Callable[[Any], str]] = {
    "search_web": search_web,
    "book_flight": book_flight,
    "find_hotel": find_hotel,
    "register_conference": register_conference,
    "generate_outline": generate_outline,
    "write_draft": write_draft,
    "review_text": review_text,
    "generate_title": generate_title,
    "summarize_text": summarize_text
}

# --- 3. The Multi-step Planning Agent ---

class MultiStepAgent:
    def __init__(self, llm_planner: Callable[[str], List[Dict[str, str]]], tools: Dict[str, Callable[[Any], str]]):
        self.llm_planner = llm_planner
        self.tools = tools
        self.context: List[str] = [] # To store observations and maintain state

    def execute_task(self, goal: str) -> str:
        print(f"\n--- Agent Initiated for Goal: '{goal}' ---")
        self.context = [] # Reset context for new goal

        # Step 1: Decompose the main goal into sub-tasks using the LLM planner
        plan = self.llm_planner(goal)
        if not plan:
            return "Failed to generate a plan. Goal might be too ambiguous."

        print("\n[Agent]: Plan generated:")
        for i, step in enumerate(plan):
            print(f"  {i+1}. {step['task']} (using tool: {step['tool']})")

        # Step 2: Execute each sub-task sequentially
        results = []
        for i, step in enumerate(plan):
            task_description = step['task']
            tool_name = step['tool']

            print(f"\n[Agent]: Executing Step {i+1}: '{task_description}'")

            if tool_name not in self.tools:
                result = f"Error: Tool '{tool_name}' not found for task '{task_description}'."
                print(f"  {result}")
                results.append(result)
                # In a real agent, this might trigger re-planning or error handling
                continue

            tool_function = self.tools[tool_name]
            
            # Dynamically call the tool based on task description
            # This is a simplified approach; a real agent would parse arguments from LLM output
            try:
                if tool_name == "book_flight":
                    # Example of specific argument parsing for a tool
                    observation = tool_function(destination="Paris", date_range="Oct 20-25, 2026")
                elif tool_name == "find_hotel":
                    observation = tool_function(location="Paris", criteria="near conference, 4-star")
                elif tool_name == "register_conference":
                    observation = tool_function(conference_name="AI Agents Summit 2026")
                elif tool_name == "generate_outline":
                    observation = tool_function(topic=task_description.replace("Outline the blog post structure", "AI Agents"))
                elif tool_name == "write_draft":
                    observation = tool_function(section=task_description.replace("Draft the ", "").replace(" sections", ""))
                elif tool_name == "generate_title":
                    observation = tool_function(content_summary="AI Agents blog post")
                elif tool_name == "review_text":
                    observation = tool_function(text="drafted content") # Placeholder
                elif tool_name == "summarize_text":
                    observation = tool_function(text="research findings") # Placeholder
                else:
                    observation = tool_function(task_description) # Generic call for other tools

                print(f"  [Observation]: {observation}")
                self.context.append(f"Step {i+1} ({task_description}): {observation}")
                results.append(observation)
            except Exception as e:
                error_msg = f"Error executing tool '{tool_name}' for task '{task_description}': {e}"
                print(f"  [Error]: {error_msg}")
                self.context.append(error_msg)
                results.append(error_msg)
                # In a robust agent, this would trigger re-planning or alternative strategies

        final_summary = "\n--- Agent Execution Complete ---\n"
        final_summary += f"Goal: '{goal}'\n"
        final_summary += "Final Context/Results:\n"
        for res in self.context:
            final_summary += f"- {res}\n"

        return final_summary

# --- Instantiate and Run the Agent ---

# Create an agent instance
agent = MultiStepAgent(llm_planner=simulate_llm_decompose_task, tools=TOOLS)

# Run the agent for a complex goal
print(agent.execute_task("Plan a trip to Paris for a conference"))

print("\n" + "="*80 + "\n")

# Run the agent for another goal
print(agent.execute_task("Write a blog post about AI agents"))

print("\n" + "="*80 + "\n")

# Run the agent for a simpler, generic goal
print(agent.execute_task("Research the latest advancements in quantum computing"))


### Interpreting the Code Output and Performance Trade-offs

The output from the code above clearly illustrates the multi-step planning process:

1.  **LLM Planning Phase**: The `[LLM Planner]` message indicates the initial decomposition of the high-level goal into a sequence of smaller, actionable sub-tasks. Notice how the simulated LLM not only breaks down the task but also suggests which `tool` is appropriate for each sub-task. This is a crucial output from modern LLMs, often in a structured format like JSON.
2.  **Agent Execution Loop**: The `[Agent]` messages show the agent iterating through each planned step. For every step, it identifies the required tool and invokes it.
3.  **Tool Invocation and Observation**: The `[Tool: <tool_name>]` messages represent the agent interacting with external systems (simulated here). The `[Observation]` messages are the results or feedback from these tool calls, which are then added to the agent's `context`. In a more advanced agent, this context would be fed back to the LLM for re-planning or to inform subsequent steps.

This sequential execution, guided by an initial plan, is the essence of multi-step planning. It allows the agent to break down a daunting problem into manageable chunks, much like a human would.

### Performance Trade-offs and Considerations (2026 Context)

While incredibly powerful, multi-step planning introduces several trade-offs:

*   **Increased Latency**: Each planning step (LLM call) and each tool invocation adds to the overall execution time. For tasks requiring real-time responses, this can be a significant bottleneck. Optimizations like parallel tool execution (where dependencies allow) or caching LLM responses for common sub-plans are critical.
*   **Cost**: LLM API calls incur costs. More steps mean more calls, leading to higher operational expenses. Efficient planning that minimizes unnecessary steps is vital.
*   **Planning Quality**: The quality of the overall solution is highly dependent on the LLM's ability to generate an effective and correct plan. Poor decomposition, incorrect tool selection, or hallucinated steps can lead to failures. Advanced LLMs in 2026 are better at this, but it's still a challenge, especially for novel or highly specialized domains.
*   **State Management**: The agent needs to maintain and update its internal state (the `context` in our example) across multiple steps. This context is crucial for the LLM to make informed decisions in subsequent planning or re-planning phases. Managing this context effectively, preventing it from becoming too large or irrelevant, is an engineering challenge.
*   **Error Handling and Re-planning**: What happens if a tool call fails? A robust agent needs mechanisms to detect failures, analyze the cause, and potentially re-plan the affected steps or even the entire strategy. This adds significant complexity to the agent's architecture.
*   **Human-in-the-Loop**: For critical or ambiguous tasks, incorporating human oversight or approval at key planning or execution points can improve reliability and trust. This is a growing trend in 2026 for complex agentic systems.

### Typical Use Cases

Multi-step planning and task decomposition are fundamental to a wide array of advanced AI applications:

*   **Automated Workflows**: Orchestrating complex business processes, from customer onboarding to supply chain management.
*   **Complex Data Analysis**: Breaking down a broad data science question into data retrieval, cleaning, analysis, visualization, and reporting steps.
*   **Scientific Discovery**: Designing experiments, analyzing results, and formulating new hypotheses.
*   **Robotics and Autonomous Systems**: Planning navigation, manipulation, and interaction sequences in dynamic environments.
*   **Software Development**: Generating code, testing, debugging, and deploying applications based on high-level requirements.
*   **Personal Assistants**: Managing schedules, booking travel, and handling complex information retrieval tasks.

As LLMs continue to improve in their reasoning and planning capabilities, the sophistication of multi-step agents will only grow, enabling them to tackle increasingly ambitious problems.


### Resources for Further Learning

To deepen your understanding of multi-step planning and agentic architectures, consider exploring these resources:

*   **Google AI Studio / Gemini API Documentation**: For integrating advanced LLMs into your agents. Understanding how to prompt for structured outputs (like JSON plans) is key.
    *   [Google AI Studio](https://aistudio.google.com/)
    *   [Gemini API Documentation](https://ai.google.dev/docs)
*   **Hugging Face Agents**: While this course focuses on building from scratch, Hugging Face provides excellent libraries and models that can be used as components for agentic systems, including tools and model hubs.
    *   [Hugging Face Transformers Agents](https://huggingface.co/docs/transformers/main/en/transformers_agents)
*   **Research Papers on LLM Planning**: Explore academic literature on how LLMs are used for planning, reasoning, and task decomposition. Keywords to search for include "LLM planning," "task decomposition with LLMs," "agentic AI architectures."
    *   A good starting point might be papers discussing frameworks like ReAct, AutoGPT, or BabyAGI, which popularized these concepts.
*   **Open-source Agent Frameworks (for inspiration)**: While we build from scratch, examining the design principles of frameworks like LangChain or LlamaIndex can provide insights into robust agent architectures.
    *   [LangChain Documentation](https://www.langchain.com/)
    *   [LlamaIndex Documentation](https://www.llamaindex.ai/)
*   **AI Ethics and Safety Guidelines**: As agents become more autonomous, understanding the ethical implications of their planning and execution is paramount.
    *   [Google's AI Principles](https://ai.google/responsibility/)

Continue to experiment with different planning strategies and tool integrations. The ability to break down and execute complex tasks is a cornerstone of truly intelligent and autonomous AI agents.
